# Brain Tumor MRI Classification using Convolutional Neural Networks

## Project Title
**Brain Tumor Detection from MRI Scans with CNNs and Transfer Learning**

## Rationale
Brain tumors are among the most critical neurological conditions; early and accurate
detection is essential for patient outcomes. MRI scans are the gold-standard imaging
modality, producing rich spatial data that CNNs are well-suited to analyse.

This project trains and evaluates:
* A **custom CNN** (built from scratch) to understand baseline deep-learning performance.
* A **transfer-learning model** (MobileNetV2 pre-trained on ImageNet) to leverage
  rich, pre-learned feature representations.

## Dataset
**Brain MRI Images for Brain Tumor Detection**
* Source: [Kaggle – navoneel/brain-mri-images-for-brain-tumor-detection](https://www.kaggle.com/datasets/navoneel/brain-mri-images-for-brain-tumor-detection)
* Two classes: `yes` (tumor present) and `no` (no tumor)
* ~253 MRI images — compact enough for rapid iteration, medically relevant.
* Grayscale / RGB JPEG images of varying sizes; resized to 224 × 224 for the CNN.

### Why this dataset is appropriate
| Criterion | Detail |
|-----------|--------|
| Public access | Free download via Kaggle API |
| CNN suitability | Natural spatial patterns (tumor mass, irregular borders) |
| Class balance | ~155 tumor / ~98 no-tumor (manageable imbalance) |
| Domain relevance | Direct medical application |

---
> **How to run**: Set `DATA_DIR` in *Section 1* to the path of the unzipped dataset, or leave
> `USE_SYNTHETIC = True` to run with procedurally-generated images for a smoke test.


## 1. Dataset Preparation + Augmentation

In [ ]:
# ── 1.1 Import Libraries ──────────────────────────────────────────────────────
import os
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from pathlib import Path
from collections import Counter

# TensorFlow / Keras
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import (
    EarlyStopping, ReduceLROnPlateau, ModelCheckpoint, TensorBoard
)
from tensorflow.keras.optimizers import Adam, SGD, RMSprop

# scikit-learn (metrics & utilities)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_curve, auc
)
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"TensorFlow version : {tf.__version__}")
print(f"GPU available      : {bool(tf.config.list_physical_devices('GPU'))}")
print("✅ All libraries imported successfully!")


In [ ]:
# ── 1.2 Global Configuration ──────────────────────────────────────────────────
# ---- Dataset settings -------------------------------------------------------
DATA_DIR      = "data/brain_tumor_dataset"   # Root folder with 'yes/' and 'no/' sub-dirs
USE_SYNTHETIC = True                          # Set False when real data is available

# ---- Image settings ---------------------------------------------------------
IMG_HEIGHT    = 224
IMG_WIDTH     = 224
IMG_SIZE      = (IMG_HEIGHT, IMG_WIDTH)
NUM_CHANNELS  = 3                             # RGB (MobileNetV2 requires 3 channels)

# ---- Training settings ------------------------------------------------------
BATCH_SIZE    = 32
EPOCHS        = 30
FINE_TUNE_EPOCHS = 10                        # Extra epochs for transfer-learning fine-tune
VALIDATION_SPLIT = 0.15
TEST_SPLIT    = 0.15

# ---- Optimizer selection (choose one: 'adam', 'sgd', 'rmsprop') -------------
OPTIMIZER_NAME = "adam"
LEARNING_RATE  = 1e-4

# ---- Output directories -----------------------------------------------------
os.makedirs("models",           exist_ok=True)
os.makedirs("reports/figures",  exist_ok=True)

CLASS_NAMES = ["no_tumor", "tumor"]   # Will be overridden when loading real data
print("✅ Configuration ready")
print(f"   Image size      : {IMG_SIZE}")
print(f"   Batch size      : {BATCH_SIZE}")
print(f"   Optimizer       : {OPTIMIZER_NAME} (lr={LEARNING_RATE})")
print(f"   Using synthetic : {USE_SYNTHETIC}")


In [ ]:
# ── 1.3 Dataset Loading Utilities ────────────────────────────────────────────

def load_real_dataset(data_dir: str, img_size: tuple) -> tuple:
    """
    Load images from a folder structure::

        data_dir/
            yes/   ← tumor images
            no/    ← non-tumor images

    Returns
    -------
    images : np.ndarray of shape (N, H, W, 3), float32 in [0, 1]
    labels : np.ndarray of shape (N,), int  (0 = no_tumor, 1 = tumor)
    class_names : list[str]
    """
    data_path   = Path(data_dir)
    class_dirs  = sorted([d for d in data_path.iterdir() if d.is_dir()])
    class_names = [d.name for d in class_dirs]

    images, labels = [], []
    for cls_idx, cls_dir in enumerate(class_dirs):
        img_paths = (
            list(cls_dir.glob("*.jpg"))
            + list(cls_dir.glob("*.jpeg"))
            + list(cls_dir.glob("*.png"))
        )
        print(f"  Class '{cls_dir.name}': {len(img_paths)} images")
        for p in img_paths:
            try:
                img = tf.keras.utils.load_img(p, target_size=img_size, color_mode="rgb")
                img = tf.keras.utils.img_to_array(img) / 255.0
                images.append(img)
                labels.append(cls_idx)
            except Exception as exc:
                print(f"    ⚠ Skipping {p.name}: {exc}")

    return np.array(images, dtype=np.float32), np.array(labels, dtype=np.int32), class_names


def create_synthetic_dataset(n_samples: int = 300, img_size: tuple = (224, 224)) -> tuple:
    """
    Generate a tiny synthetic dataset for smoke-testing the pipeline.

    Tumor images contain a bright circular region; non-tumor images do not.
    This is purely illustrative — replace with real MRI data for actual research.
    """
    np.random.seed(SEED)
    H, W = img_size
    images, labels = [], []

    for i in range(n_samples):
        img = np.random.rand(H, W, 3).astype(np.float32) * 0.3   # dark base

        if i < n_samples // 2:   # tumor class
            # Draw a bright ellipse to simulate a mass
            cy, cx  = np.random.randint(H // 4, 3 * H // 4, 2)
            ry, rx  = np.random.randint(H // 10, H // 5, 2)
            yy, xx  = np.ogrid[:H, :W]
            mask    = ((yy - cy) ** 2 / ry ** 2 + (xx - cx) ** 2 / rx ** 2) <= 1
            img[mask] = np.clip(img[mask] + 0.8, 0, 1)
            labels.append(1)
        else:
            labels.append(0)

        images.append(img)

    return (
        np.array(images, dtype=np.float32),
        np.array(labels,  dtype=np.int32),
        ["no_tumor", "tumor"],
    )


# ── Load dataset ─────────────────────────────────────────────────────────────
if USE_SYNTHETIC:
    print("Creating synthetic dataset (smoke-test mode)…")
    images, labels, CLASS_NAMES = create_synthetic_dataset(n_samples=300, img_size=IMG_SIZE)
else:
    print(f"Loading real dataset from '{DATA_DIR}' …")
    images, labels, CLASS_NAMES = load_real_dataset(DATA_DIR, IMG_SIZE)

print(f"\n✅ Dataset ready")
print(f"   Images shape : {images.shape}  dtype={images.dtype}")
print(f"   Labels shape : {labels.shape}")
print(f"   Classes      : {CLASS_NAMES}")
dist = Counter(labels)
for idx, name in enumerate(CLASS_NAMES):
    print(f"   {name}: {dist[idx]} images ({dist[idx]/len(labels)*100:.1f}%)")


In [ ]:
# ── 1.4 Train / Validation / Test Split ──────────────────────────────────────
X_train_val, X_test, y_train_val, y_test = train_test_split(
    images, labels,
    test_size=TEST_SPLIT,
    random_state=SEED, stratify=labels
)

val_ratio = VALIDATION_SPLIT / (1 - TEST_SPLIT)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val,
    test_size=val_ratio,
    random_state=SEED, stratify=y_train_val
)

print("Data split summary")
print(f"  Train      : {X_train.shape[0]:>4} samples")
print(f"  Validation : {X_val.shape[0]:>4} samples")
print(f"  Test       : {X_test.shape[0]:>4} samples")


### 1.5 Data Augmentation

To improve generalisation and combat the small dataset size we apply the following
random transformations **only to the training set**:

| Transform | Range |
|-----------|-------|
| Rotation | ±20° |
| Width / Height shift | ±10% |
| Zoom | ±15% |
| Horizontal flip | True |
| Brightness adjustment | ±20% |

Validation and test images receive **no** augmentation (only normalisation).


In [ ]:
# ── 1.5 Augmentation with Keras ImageDataGenerator ───────────────────────────
# Training generator — augmentation applied
train_datagen = ImageDataGenerator(
    rotation_range=20,
    width_shift_range=0.10,
    height_shift_range=0.10,
    shear_range=0.10,
    zoom_range=0.15,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode="nearest",
)

# Validation / test generators — no augmentation
val_datagen  = ImageDataGenerator()
test_datagen = ImageDataGenerator()

# Create tf.data-compatible generators (flow from numpy arrays)
train_gen = train_datagen.flow(
    X_train, y_train, batch_size=BATCH_SIZE, seed=SEED, shuffle=True
)
val_gen   = val_datagen.flow(
    X_val, y_val, batch_size=BATCH_SIZE, seed=SEED, shuffle=False
)
test_gen  = test_datagen.flow(
    X_test, y_test, batch_size=BATCH_SIZE, seed=SEED, shuffle=False
)

print("✅ Data generators created")


# ── Visualise augmented samples ───────────────────────────────────────────────
def visualise_augmented_samples(generator, class_names, n_cols=5, n_rows=2):
    """Show a grid of augmented training images."""
    batch_imgs, batch_lbls = next(generator)
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(3 * n_cols, 3 * n_rows))
    for i, ax in enumerate(axes.ravel()):
        if i >= len(batch_imgs):
            ax.axis("off")
            continue
        ax.imshow(batch_imgs[i])
        ax.set_title(class_names[int(batch_lbls[i])], fontsize=9)
        ax.axis("off")
    plt.suptitle("Augmented Training Samples", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.savefig("reports/figures/augmented_samples.png", dpi=150, bbox_inches="tight")
    plt.show()

# Reset generator before sampling
sample_gen = train_datagen.flow(X_train, y_train, batch_size=10, seed=SEED)
visualise_augmented_samples(sample_gen, CLASS_NAMES)


---
## 2. CNN Model Design

Two architectures are defined:

1. **Custom CNN** — built from scratch using stacked Conv-BN-MaxPool blocks followed by
   fully-connected layers. Gives full control and interpretability.
2. **Transfer Learning (MobileNetV2)** — a lightweight ImageNet-pretrained backbone with
   a custom classification head. Excels when labelled data are scarce.

Both models produce a single sigmoid output (binary classification).


In [ ]:
# ── 2.1 Custom CNN Model ──────────────────────────────────────────────────────
def build_custom_cnn(
    input_shape: tuple = (224, 224, 3),
    num_classes: int   = 1,
    dropout_rate: float = 0.5,
) -> Model:
    """
    Custom Convolutional Neural Network.

    Architecture
    ------------
    Input → [Conv2D → BN → ReLU → MaxPool] × 4 → GAP → Dense → Dropout → Output

    Parameters
    ----------
    input_shape : (H, W, C)
    num_classes : 1 for binary (sigmoid), >1 for multi-class (softmax)
    dropout_rate : dropout probability before the final dense layer
    """
    inputs = Input(shape=input_shape, name="input_image")

    x = inputs
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, (3, 3), padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.Conv2D(filters, (3, 3), padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(512, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(dropout_rate / 2)(x)

    if num_classes == 1:
        outputs = layers.Dense(1, activation="sigmoid", name="output")(x)
    else:
        outputs = layers.Dense(num_classes, activation="softmax", name="output")(x)

    return Model(inputs, outputs, name="Custom_CNN")


custom_cnn = build_custom_cnn(input_shape=(IMG_HEIGHT, IMG_WIDTH, NUM_CHANNELS))
custom_cnn.summary()


In [ ]:
# ── 2.2 Transfer-Learning Model (MobileNetV2) ─────────────────────────────────
def build_transfer_model(
    input_shape: tuple  = (224, 224, 3),
    num_classes: int    = 1,
    dropout_rate: float = 0.4,
) -> tuple:
    """
    Transfer-Learning model with MobileNetV2 backbone (pre-trained on ImageNet).

    Strategy
    --------
    Phase 1 — freeze the backbone, train only the classification head.
    Phase 2 — unfreeze layers from `FINE_TUNE_AT` onward and fine-tune at low LR.

    Parameters
    ----------
    input_shape  : (H, W, C) — must be ≥ (96, 96, 3) for MobileNetV2
    num_classes  : 1 for binary sigmoid output
    dropout_rate : dropout before the final dense layer

    Returns
    -------
    (model, base_model) — Keras Model and a direct reference to the MobileNetV2
    backbone, used later to selectively unfreeze layers for fine-tuning.
    """
    base_model = MobileNetV2(
        input_shape=input_shape,
        include_top=False,
        weights="imagenet",
        name="mobilenetv2_backbone",
    )
    base_model.trainable = False   # Freeze during Phase 1

    inputs = Input(shape=input_shape, name="input_image")

    # MobileNetV2 expects inputs pre-processed; apply its preprocessing
    x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(dropout_rate)(x)

    if num_classes == 1:
        outputs = layers.Dense(1, activation="sigmoid", name="output")(x)
    else:
        outputs = layers.Dense(num_classes, activation="softmax", name="output")(x)

    model = Model(inputs, outputs, name="MobileNetV2_Transfer")
    return model, base_model


transfer_model, mobilenet_base = build_transfer_model(
    input_shape=(IMG_HEIGHT, IMG_WIDTH, NUM_CHANNELS)
)

# Count trainable vs frozen params
total     = sum(v.numpy().size for v in transfer_model.trainable_weights)
non_train = sum(
    v.numpy().size for v in transfer_model.non_trainable_weights
)
print(f"Trainable params     : {total:,}")
print(f"Non-trainable params : {non_train:,}")
transfer_model.summary()


---
## 3. Model Training

### Selectable Optimizer
The `OPTIMIZER_NAME` variable (set in Section 1) controls which optimizer is used.
Supported values: `'adam'`, `'sgd'`, `'rmsprop'`.

### Callbacks
| Callback | Purpose |
|----------|---------|
| `EarlyStopping` | Stop when val_loss stagnates (patience=7) |
| `ReduceLROnPlateau` | Halve LR when val_loss plateaus (patience=3) |
| `ModelCheckpoint` | Save the best model weights |

### Transfer-Learning Training Protocol
1. **Phase 1** — train only the classification head (backbone frozen).
2. **Phase 2** — unfreeze upper backbone layers and fine-tune at a 10× smaller LR.


In [ ]:
# ── 3.1 Optimizer Factory ─────────────────────────────────────────────────────
def get_optimizer(name: str, lr: float):
    """Return a Keras optimizer by name."""
    name = name.lower()
    if name == "adam":
        return Adam(learning_rate=lr)
    elif name == "sgd":
        return SGD(learning_rate=lr, momentum=0.9, nesterov=True)
    elif name == "rmsprop":
        return RMSprop(learning_rate=lr)
    else:
        raise ValueError(f"Unknown optimizer '{name}'. Choose: adam | sgd | rmsprop")


def get_callbacks(model_name: str, monitor: str = "val_loss") -> list:
    """Standard training callbacks."""
    return [
        EarlyStopping(
            monitor=monitor, patience=7,
            restore_best_weights=True, verbose=1
        ),
        ReduceLROnPlateau(
            monitor=monitor, factor=0.5, patience=3,
            min_lr=1e-7, verbose=1
        ),
        ModelCheckpoint(
            filepath=f"models/{model_name}_best.keras",
            monitor=monitor, save_best_only=True, verbose=0
        ),
    ]


optimizer  = get_optimizer(OPTIMIZER_NAME, LEARNING_RATE)
loss_fn    = "binary_crossentropy"
metrics    = ["accuracy"]

print(f"Optimizer : {optimizer.__class__.__name__}  lr={LEARNING_RATE}")
print(f"Loss      : {loss_fn}")


In [ ]:
# ── 3.2 Train Custom CNN ──────────────────────────────────────────────────────
print("=" * 60)
print("TRAINING — Custom CNN")
print("=" * 60)

custom_cnn.compile(
    optimizer=get_optimizer(OPTIMIZER_NAME, LEARNING_RATE),
    loss=loss_fn,
    metrics=metrics,
)

history_custom = custom_cnn.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=get_callbacks("custom_cnn"),
    verbose=1,
)

print("\n✅ Custom CNN training complete!")


In [ ]:
# ── 3.3 Train Transfer-Learning Model (Phase 1 + Phase 2) ────────────────────
print("=" * 60)
print("TRAINING — MobileNetV2 Transfer Learning")
print("=" * 60)

# ---- Phase 1: frozen backbone -----------------------------------------------
print("\n--- Phase 1: Training classification head (backbone frozen) ---")
transfer_model.compile(
    optimizer=get_optimizer(OPTIMIZER_NAME, LEARNING_RATE),
    loss=loss_fn,
    metrics=metrics,
)

history_tl_p1 = transfer_model.fit(
    train_gen,
    epochs=EPOCHS,
    validation_data=val_gen,
    callbacks=get_callbacks("transfer_phase1"),
    verbose=1,
)

# ---- Phase 2: fine-tune upper backbone layers --------------------------------
print("\n--- Phase 2: Fine-tuning upper backbone layers ---")

# Unfreeze the backbone using the named reference returned by build_transfer_model()
mobilenet_base.trainable = True

# Freeze all layers below FINE_TUNE_AT (set in Section 1 config), unfreeze above
for layer in mobilenet_base.layers[:FINE_TUNE_AT]:
    layer.trainable = False

transfer_model.compile(
    optimizer=get_optimizer(OPTIMIZER_NAME, LEARNING_RATE / 10),  # 10× smaller LR
    loss=loss_fn,
    metrics=metrics,
)

history_tl_p2 = transfer_model.fit(
    train_gen,
    epochs=FINE_TUNE_EPOCHS,
    validation_data=val_gen,
    callbacks=get_callbacks("transfer_phase2"),
    verbose=1,
)

print("\n✅ Transfer-learning training complete!")


---
## 4. Performance Evaluation

We evaluate both models on the **held-out test set** using:

* Accuracy, Precision, Recall, F1-Score
* Confusion Matrix (with class labels)
* Training / Validation Accuracy & Loss Curves
* ROC Curve and AUC score


In [ ]:
# ── 4.1 Evaluation Utilities ─────────────────────────────────────────────────
def evaluate_model(model, X: np.ndarray, y_true: np.ndarray,
                   class_names: list, model_label: str,
                   threshold: float = 0.5) -> dict:
    """
    Compute and print classification metrics for a binary Keras model.

    Returns a dict with keys: accuracy, precision, recall, f1,
    roc_auc, fpr, tpr, y_pred, y_prob, confusion_matrix.
    """
    y_prob = model.predict(X, verbose=0).ravel()
    y_pred = (y_prob >= threshold).astype(int)

    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
    cm   = confusion_matrix(y_true, y_pred)

    fpr, tpr, _ = roc_curve(y_true, y_prob)
    roc_auc     = auc(fpr, tpr)

    print(f"\n{'='*55}")
    print(f"  {model_label}")
    print(f"{'='*55}")
    print(f"  Accuracy  : {acc:.4f}")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}")
    print(f"  F1-Score  : {f1:.4f}")
    print(f"  ROC-AUC   : {roc_auc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names))

    return {
        "accuracy": acc, "precision": prec,
        "recall": rec, "f1": f1,
        "roc_auc": roc_auc, "fpr": fpr, "tpr": tpr,
        "y_pred": y_pred, "y_prob": y_prob,
        "confusion_matrix": cm,
    }


# ── Evaluate both models ──────────────────────────────────────────────────────
results_custom   = evaluate_model(custom_cnn,   X_test, y_test, CLASS_NAMES, "Custom CNN")
results_transfer = evaluate_model(transfer_model, X_test, y_test, CLASS_NAMES, "MobileNetV2 Transfer")


In [ ]:
# ── 4.2 Confusion Matrices ────────────────────────────────────────────────────
def plot_confusion_matrix(cm: np.ndarray, class_names: list,
                          title: str, ax: plt.Axes):
    """Plot a single normalised confusion matrix on `ax`."""
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(
        cm_norm, annot=cm, fmt="d", cmap="Blues",
        xticklabels=class_names, yticklabels=class_names,
        linewidths=0.5, ax=ax, cbar=False,
        annot_kws={"size": 13},
    )
    ax.set_title(title, fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted", fontsize=11)
    ax.set_ylabel("Actual",    fontsize=11)


fig, axes = plt.subplots(1, 2, figsize=(11, 4))
plot_confusion_matrix(results_custom["confusion_matrix"],   CLASS_NAMES, "Custom CNN",            axes[0])
plot_confusion_matrix(results_transfer["confusion_matrix"], CLASS_NAMES, "MobileNetV2 Transfer", axes[1])
plt.suptitle("Confusion Matrices — Test Set", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("reports/figures/confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 4.3 Training / Validation Curves ─────────────────────────────────────────
def merge_histories(*histories):
    """Concatenate Keras History objects (useful for Phase 1 + Phase 2)."""
    merged = {"loss": [], "val_loss": [], "accuracy": [], "val_accuracy": []}
    for h in histories:
        for key in merged:
            if key in h.history:
                merged[key].extend(h.history[key])
    return merged


def plot_training_curves(history_dict: dict, title: str, ax_loss: plt.Axes, ax_acc: plt.Axes):
    """Plot loss and accuracy curves side-by-side."""
    epochs = range(1, len(history_dict["loss"]) + 1)

    ax_loss.plot(epochs, history_dict["loss"],     label="Train Loss", linewidth=2)
    ax_loss.plot(epochs, history_dict["val_loss"],  label="Val Loss",   linewidth=2, linestyle="--")
    ax_loss.set_title(f"{title} — Loss",     fontsize=11, fontweight="bold")
    ax_loss.set_xlabel("Epoch"); ax_loss.set_ylabel("Loss")
    ax_loss.legend(); ax_loss.grid(True, alpha=0.3)

    ax_acc.plot(epochs, history_dict["accuracy"],     label="Train Acc", linewidth=2)
    ax_acc.plot(epochs, history_dict["val_accuracy"], label="Val Acc",   linewidth=2, linestyle="--")
    ax_acc.set_title(f"{title} — Accuracy", fontsize=11, fontweight="bold")
    ax_acc.set_xlabel("Epoch"); ax_acc.set_ylabel("Accuracy")
    ax_acc.legend(); ax_acc.grid(True, alpha=0.3)


fig, axes = plt.subplots(2, 2, figsize=(14, 9))

plot_training_curves(history_custom.history,              "Custom CNN",            axes[0][0], axes[0][1])
plot_training_curves(merge_histories(history_tl_p1, history_tl_p2),
                     "MobileNetV2 Transfer", axes[1][0], axes[1][1])

plt.suptitle("Training & Validation Curves", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("reports/figures/training_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 4.4 ROC Curves ────────────────────────────────────────────────────────────
plt.figure(figsize=(8, 6))

for label, res, ls in [
    ("Custom CNN",            results_custom,   "-"),
    ("MobileNetV2 Transfer",  results_transfer, "--"),
]:
    plt.plot(
        res["fpr"], res["tpr"],
        label=f"{label}  (AUC = {res['roc_auc']:.3f})",
        linewidth=2, linestyle=ls,
    )

plt.plot([0, 1], [0, 1], "k:", linewidth=1, label="Random Classifier")
plt.xlim([-0.01, 1.01]); plt.ylim([-0.01, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate",  fontsize=12)
plt.title("ROC Curves — Test Set", fontsize=14, fontweight="bold")
plt.legend(loc="lower right", fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("reports/figures/roc_curves.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# ── 4.5 Summary Metrics Table ────────────────────────────────────────────────
summary = pd.DataFrame(
    {
        "Custom CNN":           {k: v for k, v in results_custom.items()
                                 if k in ("accuracy","precision","recall","f1","roc_auc")},
        "MobileNetV2 Transfer": {k: v for k, v in results_transfer.items()
                                 if k in ("accuracy","precision","recall","f1","roc_auc")},
    }
).T.round(4)

print("\n" + "=" * 50)
print("  FINAL TEST-SET METRICS COMPARISON")
print("=" * 50)
print(summary.to_string())
summary.to_csv("reports/model_comparison.csv")
print("\n✅ Metrics saved to reports/model_comparison.csv")

best_name = summary["f1"].idxmax()
print(f"\n🏆 Best model by F1-Score : {best_name}  (F1 = {summary.loc[best_name,'f1']:.4f})")


In [ ]:
# ── 4.6 Save Models ──────────────────────────────────────────────────────────
custom_cnn.save("models/custom_cnn_final.keras")
transfer_model.save("models/transfer_mobilenetv2_final.keras")
print("✅ Models saved to models/")


---
## 5. Experimental Plan and Analysis Outline

Use this protocol to systematically compare approaches and report results.

### 5.1 Baseline vs Transfer Learning Comparison

| Experiment | Model | Notes |
|-----------|-------|-------|
| E1 | Custom CNN (Adam, lr=1e-4) | Baseline |
| E2 | Custom CNN (SGD+momentum, lr=1e-3) | Optimizer ablation |
| E3 | MobileNetV2 — head-only (Adam, lr=1e-4) | Phase 1 only |
| E4 | MobileNetV2 — fine-tuned (Adam, lr=1e-5) | Phase 1 + Phase 2 |
| E5 | MobileNetV2 — fine-tuned (SGD, lr=1e-4) | Optimizer ablation |

> **Tip**: Change `OPTIMIZER_NAME` and `LEARNING_RATE` at the top of Section 1 and
> re-run Sections 3-4 for each experiment row.

### 5.2 Hyperparameter Tuning Checklist
- [ ] Learning rate: try `[1e-3, 1e-4, 1e-5]`
- [ ] Batch size: try `[16, 32, 64]`
- [ ] Dropout rate: try `[0.3, 0.5, 0.7]`
- [ ] Fine-tune-at layer: try `[80, 100, 120]` for MobileNetV2

### 5.3 Augmentation Ablation
Run with augmentation **on** vs **off** (`USE_AUGMENTATION = False` variant) and
compare test accuracy to quantify the regularisation benefit.

### 5.4 Reporting Metrics
For each experiment record:

| Metric | Why it matters |
|--------|---------------|
| Test Accuracy | Overall correctness |
| Precision | How many predicted tumors are real tumors |
| Recall (Sensitivity) | How many real tumors are caught — critical in medicine |
| F1-Score | Harmonic mean; balances precision & recall |
| ROC-AUC | Threshold-independent discrimination ability |
| Training time | Practical deployment consideration |

### 5.5 Suggested Report Outline
1. Introduction — clinical motivation, dataset description
2. Methods — preprocessing, augmentation, model architectures, training protocol
3. Results — tables + figures for each experiment (curves, confusion matrices, ROC)
4. Discussion — which model is best and why, failure modes (false negatives are dangerous)
5. Conclusion — key findings, limitations, future work (e.g., Grad-CAM visualisations,
   larger dataset, multi-class tumour grading)

### 5.6 Bonus: Grad-CAM Visualisation (recommended addition)
Grad-CAM highlights the image regions that drove a prediction, making the model
interpretable to clinicians. A minimal implementation:

```python
import tensorflow as tf

def grad_cam(model, img_array, layer_name, class_idx=0):
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array[np.newaxis])
        loss = predictions[:, class_idx]
    grads      = tape.gradient(loss, conv_outputs)[0]
    weights    = tf.reduce_mean(grads, axis=(0, 1))
    cam        = tf.reduce_sum(tf.multiply(weights, conv_outputs[0]), axis=-1)
    cam        = tf.nn.relu(cam).numpy()
    cam        = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
    return cam
```


In [ ]:
# ── Final Project Summary ─────────────────────────────────────────────────────
print("""
╔══════════════════════════════════════════════════════════╗
║   Brain Tumor MRI Classification — Project Summary       ║
╠══════════════════════════════════════════════════════════╣
║  Dataset  : Brain MRI Images (yes/no tumor)              ║
║  Input    : {h}×{w}×{c} RGB MRI images                       ║
║  Models   : Custom CNN  |  MobileNetV2 Transfer Learning ║
║  Task     : Binary classification (tumor / no-tumor)     ║
╠══════════════════════════════════════════════════════════╣
║  Sections completed                                      ║
║  ✅ 1. Dataset Preparation + Augmentation                ║
║  ✅ 2. CNN Model Design (custom + transfer learning)     ║
║  ✅ 3. Model Training (selectable optimizer)             ║
║  ✅ 4. Performance Evaluation (acc, prec, rec, F1,       ║
║         confusion matrix, ROC, training curves)          ║
║  ✅ 5. Experimental Plan                                 ║
╠══════════════════════════════════════════════════════════╣
║  Output files                                            ║
║  • models/custom_cnn_final.keras                        ║
║  • models/transfer_mobilenetv2_final.keras              ║
║  • reports/model_comparison.csv                         ║
║  • reports/figures/augmented_samples.png                ║
║  • reports/figures/confusion_matrices.png               ║
║  • reports/figures/training_curves.png                  ║
║  • reports/figures/roc_curves.png                       ║
╚══════════════════════════════════════════════════════════╝
""".format(h=IMG_HEIGHT, w=IMG_WIDTH, c=NUM_CHANNELS))
